# 02 Statistical Analysis

Deep-dive into the CheXpert label distributions, uncertainty rates, and class-imbalance
statistics needed to justify modelling decisions (loss weighting, label selection, etc.).

**Depends on:** `notebooks/01_data_exploration.ipynb` having been run (parquet manifests must exist).

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

_cwd = Path().resolve()
REPO_ROOT = next(
    (p for p in [_cwd, *_cwd.parents] if (p / 'pyproject.toml').exists()),
    _cwd
)
TRAIN_PARQUET = REPO_ROOT / 'src' / 'data' / 'train_manifest.parquet'
VALID_PARQUET = REPO_ROOT / 'src' / 'data' / 'valid_manifest.parquet'

train_df = pl.read_parquet(TRAIN_PARQUET)
valid_df = pl.read_parquet(VALID_PARQUET)
print(f'Train: {train_df.shape} | Valid: {valid_df.shape}')

## 1. Uncertainty Rate per Label
How often does each label carry a `-1` (uncertain) annotation before U-Zero policy?

In [ ]:
UNCERTAIN_COLS = [c for c in train_df.columns if c.endswith('_uncertain')]
n = len(train_df)

uncertain_stats = [
    {'label': c.replace('_uncertain', ''), 'pct': 100 * train_df[c].sum() / n}
    for c in UNCERTAIN_COLS if c in train_df.columns
]
uncertain_stats.sort(key=lambda x: -x['pct'])

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh([d['label'] for d in uncertain_stats], [d['pct'] for d in uncertain_stats], color='salmon')
ax.set_xlabel('Uncertainty rate (%)')
ax.set_title('Train — Fraction of samples originally labelled as uncertain (-1)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 2. Positive-Weight Calculation for BCEWithLogitsLoss

In [ ]:
CLINICAL_12 = [
    'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion',
    'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
    'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture',
]

print(f"{'Label':<35} {'Pos':>8} {'Neg':>8} {'Neg:Pos':>10} {'pos_weight':>12}")
print('-' * 78)
for col in CLINICAL_12:
    if col not in train_df.columns:
        continue
    pos = int(train_df[col].sum())
    neg = n - pos
    ratio = neg / pos if pos > 0 else float('inf')
    print(f"{col:<35} {pos:>8,} {neg:>8,} {ratio:>10.1f} {ratio:>12.2f}")

## 3. Train vs Validation Distribution Test (Chi-Square)
Check whether the label distributions are statistically consistent across splits.

In [ ]:
from scipy.stats import chi2_contingency

print(f"{'Label':<35} {'chi2':>8} {'p-value':>10} {'Consistent?':>12}")
print('-' * 70)
for col in CLINICAL_12:
    if col not in train_df.columns or col not in valid_df.columns:
        continue
    train_pos = int(train_df[col].sum())
    valid_pos = int(valid_df[col].sum())
    contingency = [[train_pos, len(train_df) - train_pos],
                   [valid_pos, len(valid_df) - valid_pos]]
    chi2, p, *_ = chi2_contingency(contingency)
    flag = 'YES' if p > 0.05 else 'NO  <--'
    print(f"{col:<35} {chi2:>8.2f} {p:>10.4f} {flag:>12}")